# Sprint 4 Live — Companion UI and integration

Use this notebook with **LS13–LS14**. Your required work happens in the local UI and service. The notebook provides a planning/evidence card and an **optional direct API observer**; it cannot prove that a browser form or connector works.

| Session | Learning outcome | Work to show |
|---|---|---|
| LS13 | Build a scoped companion UI using Lovable (Supabase backend built in — no separate setup): one input form, one response display, one logged interaction. Richer dashboards are extension tasks, not required for credit. | Your generated form, actual response state, and its matching safe request record. |
| LS14 | Run end-to-end tests across UI → service → model and write the README and API documentation. | A real normal path, a controlled failure, repaired instructions and a reader retry. |

Keep the scope to one interaction. Start with the [Lovable student guide](https://github.com/richhiey/ai-app-dev_Mod-B/blob/main/docs/lovable-student-guide.md), [local connector](https://github.com/richhiey/ai-app-dev_Mod-B/blob/main/examples/lovable/README.md), and [recovery reference](https://github.com/richhiey/ai-app-dev_Mod-B/blob/main/examples/lovable/reference-service.md). The reference is hand-authored support, not evidence of your Lovable build.

**How to use this notebook:** save a personal copy. Follow the LS13/LS14 sections with the instructor; do not run every cell automatically. The optional network section stays disabled until a facilitator supplies a hosted HTTPS service. No public deployment, tunnel or separate Supabase project is required. A hosted notebook cannot reach your laptop's localhost.

## LS13 — Build one useful interaction

A form collects input; the UI's server connector supplies the caller credential; FieldCare owns model access. Your browser should send to same-origin `/api/fieldcare`. Keep private values on the server.

1. Generate one question form and response display in Lovable. Inspect its real framework and submit handler before adapting the supplied connector. Sync the UI to its separate repository and follow that project's startup instructions.
2. Run the UI and service locally. Map `question` and optional `equipment_id`; omit a blank optional identifier. Clear the previous answer and request ID before another submission.
3. Predict the visible state, then submit. During a stream, `delta` adds text; only `complete` establishes completion. An `error` or missing terminal event leaves partial text unfinished. JSON clarification is a request for context, not a generated answer.
4. Find this interaction's exact request ID in the service's safe record. Check its route and outcome. The provider-backed path still needs a cited-claim check.

**Check:** Can a peer follow your form → connector → service → response and match the same request ID? A mock preview proves screen behavior only.

**Independent application · 12 minutes:** improve one field explanation or incomplete-answer message without changing the API contract. Repeat the interaction and justify the change from what you observed. Keep the source change, actual result and safe request reference. Dashboards are optional extensions.

## LS14 — Test the path and the instructions

1. Follow your own README to start the two apps. Submit a supported request and record the actual terminal state. A running process or health response does not establish model generation.
2. Leave the UI running, stop only your identified service process, and submit once. Record the connector error and UI state. Restart the service and repeat the original request. Do not use this restart as evidence that a rate-limit window expired.
3. Ask a peer to follow the tested path using your documentation. Record the first missing or ambiguous instruction; switch roles so each person receives a reader attempt.
4. Revise the service README, API documentation and UI run instructions. Retry the repaired instruction. Distinguish a retry on an installed environment from a full clean-install test.

**Independent application · 12 minutes:** make one evidence-based improvement in each document and retain a reader retry. Describe unsupported/provider-unavailable checks as pending.

Copy this card into your own notebook notes and complete it from your work:

| Record | Your observation |
|---|---|
| UI/source revision and input class | |
| Normal path: state and request ID | |
| Matching safe service record | |
| Controlled failure: expected versus actual | |
| Documentation paths and repaired instruction | |
| Reader retry result and remaining check | |

Keep keys, prompts containing personal data and raw service logs out of the card. Your local UI evidence remains separate from the optional API calls below.

## Optional API observation — setup once

Use this only when a facilitator already supplies a reachable HTTPS service. Leave `RUN_HOSTED_OBSERVER = False` to use the planning card without making any service requests. No provider key belongs here.

The setup clones course `main` and installs its locked dependencies. It prints the source revision so you can identify the runtime used. On Colab, use a Python CPU runtime and allow setup to finish. For local Jupyter use the course Python environment. A failed installation must be repaired before continuing; do not treat later import errors as API failures.

For hosted calls, put `FIELDCARE_SERVICE_URL` and `FIELDCARE_CALLER_KEY` in Colab Secrets and grant notebook access. In local Jupyter supply the same names privately as environment variables. Use only the facilitator's origin, with no route or query. Enable the switch, then rerun setup. Missing settings produce a clear error without printing their values.

In [ ]:
from pathlib import Path
import os, sys, subprocess, tempfile, json, time

REPOSITORY = "https://github.com/richhiey/ai-app-dev_Mod-B.git"
REPO = Path(tempfile.mkdtemp(prefix="module-b-live-observer-source-")) / "repo"
subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", REPOSITORY, str(REPO)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "-c", str(REPO / "requirements.lock")], check=True)
previous_helper_path = globals().get("LIVE_HELPER_PATH")
import importlib
if previous_helper_path in sys.path:
    sys.path.remove(previous_helper_path)
LIVE_HELPER_PATH = str(REPO / "src")
sys.path.insert(0, LIVE_HELPER_PATH)
# Select the new checkout even when setup is rerun in the same runtime.
for module_name in tuple(sys.modules):
    if module_name == "module_b" or module_name.startswith("module_b."):
        del sys.modules[module_name]
importlib.invalidate_caches()
import requests
from module_b.live_observer import hosted_settings, observe_response, transport_failure

SOURCE_REVISION = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
boundary_observations = []
supported_observation = None
attempts = []
RUN_HOSTED_OBSERVER = False
SERVICE_URL, CALLER_KEY = None, None
if RUN_HOSTED_OBSERVER:
    SERVICE_URL, CALLER_KEY = hosted_settings()
print("Course source revision:", SOURCE_REVISION)
print("Hosted observer enabled:", RUN_HOSTED_OBSERVER, "— private settings are not displayed.")

### Predict three caller boundaries

A caller credential identifies the API caller. Request validation checks the input shape. Clarification asks for context after validation. With the supplied course service and available allowance, predict `401` for no key, `200` with `needs_clarification` for a question without equipment, and `422` for a missing required question.

These expectations are conditional: a wrong key may give `401`; exhausted allowance may give `429` before input validation. Read `Retry-After`, wait while the same service keeps running, and repeat only the relevant call. The two authenticated attempts consume allowance. Do not replace an observed result with the predicted number.

Run the next cell to see the **actual HTTP call**. The helper reads its response, preserves request ID, measures time and reports stream completion. It stops on a terminal event and does not infer provider provenance from text. Timeouts use finite connect/read limits plus an observation budget.

In [ ]:
boundary_observations = []  # Rerunning replaces this cell's earlier observations.
if RUN_HOSTED_OBSERVER:
    cases = [
        ("missing caller credential", {"question": "Which filter checks are documented?"}, False),
        ("valid input missing optional context", {"question": "Which airflow checks are documented?"}, True),
        ("missing required question", {}, True),
    ]
    for label, payload, authenticate in cases:
        headers = {"Accept": "application/x-ndjson, application/json"}
        if authenticate:
            headers["X-API-Key"] = CALLER_KEY
        started = time.monotonic()
        try:
            with requests.post(
                f"{SERVICE_URL}/v1/diagnose-stream", json=payload,
                headers=headers, stream=True, timeout=(5, 45), allow_redirects=False,
            ) as response:
                observation = observe_response(response, label=label, started=started)
        except requests.RequestException as error:
            observation = transport_failure(label, error)
        boundary_observations.append(observation)
        print({key: observation[key] for key in
               ("label", "http_status", "application_status", "request_id", "retry_after", "failure")})
else:
    print("Skipped: no hosted requests. Continue with your local UI evidence card.")

### Observe one supported request — explicit opt-in

Set `RUN_SUPPORTED_REQUEST = True` only after reviewing the boundary calls, the available quota, and the facilitator's provider preparation. This request can consume embedding/generation quota. Expected behavior on a prepared provider service is metadata, progressive content, and a terminal `complete` event; wording and timings vary.

Read the reported `mode`. A fixture can stream fixed text and complete without a model call. `live` is a service label; establish successful provider work by the same run's matching service record and a cited-claim check. A failure or missing terminal event is not a completed answer. This direct call still does not validate your browser or connector.

In [ ]:
RUN_SUPPORTED_REQUEST = False
supported_observation = None  # Clear stale evidence even when this run is skipped.
if RUN_HOSTED_OBSERVER and RUN_SUPPORTED_REQUEST:
    payload = {"question": "Which filter and airflow checks are documented?", "equipment_id": "EQ-FC-1002"}
    started = time.monotonic()
    try:
        with requests.post(
            f"{SERVICE_URL}/v1/diagnose-stream", json=payload,
            headers={"Accept": "application/x-ndjson, application/json", "X-API-Key": CALLER_KEY},
            stream=True, timeout=(5, 45), allow_redirects=False,
        ) as response:
            supported_observation = observe_response(response, label="supported request", started=started)
    except requests.RequestException as error:
        supported_observation = transport_failure("supported request", error)
    print(supported_observation)
else:
    print("Skipped: supported/provider observation remains pending.")

### Check the evidence and recover deliberately

| Observation | Meaning and next action |
|---|---|
| `401` | Confirm the assigned caller setting privately. Do not print its value. |
| `422` | Compare required fields/types with the API contract; keep the deliberately invalid case labelled. |
| `429` | Read `Retry-After`; wait before repeating a single request. Earlier calls consumed allowance. |
| Timeout or connection error | Check the facilitator's hosted endpoint. Laptop localhost is not reachable from Colab. Keep the failed attempt. |
| HTML or unexpected content type | Check the service origin; a website/login page is not the API. Redirects are deliberately not followed. |
| `error`, interrupted stream or malformed event | Preserve the failure category; partial text is unfinished. Match its request ID if available. |
| `mode=fixture` | Protocol rehearsal only; no evidence of retrieval or generation. |
| Runtime restarted | Rerun setup and reload private settings; earlier in-memory observations are gone. Your downloaded record remains separate. |

Compare your prediction with each actual result. Write one supported conclusion and one question that requires a different test. Keep skipped checks pending.

In [ ]:
attempts = list(boundary_observations)
if supported_observation is not None:
    attempts.append(supported_observation)
for attempt in attempts:
    print(attempt)
if not attempts:
    print("No optional API observations were executed. This does not change your local UI evidence.")

## Save your work

Save your notebook copy and the local UI/documentation work separately. The optional JSON below records only request metadata, outcomes and timing from this run. It excludes keys, request bodies, generated text, citations and raw headers. A record with zero attempts explicitly means **not executed**.

Run the export cell once at the end. Colab downloads the file; local Jupyter prints its path. Review the file before sharing. Your own UI request ID and matching safe record belong in the LS13/LS14 card, not in these API observations.

## Summary

You can explain the UI-to-service boundary, show an interaction with its own trace, and give a reader tested instructions. Optional API observations help isolate service behavior; the local UI's own run establishes the required integration.

In [ ]:
from datetime import datetime, timezone

attempts = list(boundary_observations)
if supported_observation is not None:
    attempts.append(supported_observation)

live_evidence = {
    "captured_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_revision": SOURCE_REVISION,
    "notebook_is_direct_api_caller": True,
    "observation_status": "executed" if attempts else "not_executed",
    "attempts": attempts,
    "supported_request_executed": supported_observation is not None,
    "ui_evidence_required_separately": True,
}
evidence_path = Path.cwd() / f"sprint-4-live-api-evidence-{time.time_ns()}.json"
evidence_path.write_text(json.dumps(live_evidence, indent=2) + "\n", encoding="utf-8")
try:
    from google.colab import files
except ImportError:
    print("Saved optional API evidence:", evidence_path)
else:
    files.download(str(evidence_path))